# DANTE 合金材料设计优化

本笔记本演示如何使用DANTE框架进行合金材料的成分优化，以获取最佳的机械性能（弹性模量和屈服强度的组合）。

## 内容概览

1. **第一部分**：数据加载与预处理
2. **第二部分**：定义DANTE算法组件
3. **第三部分**：构建神经网络代理模型
4. **第四部分**：使用DANTE进行优化
5. **第五部分**：结果可视化与分析

## 第一部分：数据加载与预处理

首先导入必要的库，并加载合金材料数据集。

In [ ]:
    # 导入必要的库
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 设置可视化样式
plt.style.use('ggplot')
sns.set(style="whitegrid")

# 检查数据文件是否存在
data_path = "data.csv"
if os.path.exists(data_path):
    print(f"数据文件 {data_path} 存在")
else:
    print(f"警告：数据文件 {data_path} 不存在！")
    
    # 如果在上级目录中有数据文件，尝试复制它
    parent_data_path = "../../../data.csv"
    if os.path.exists(parent_data_path):
        print(f"在上级目录中找到数据文件，正在复制到当前目录...")
        import shutil
        shutil.copy(parent_data_path, data_path)
        print("复制完成！")
    else:
        print("在上级目录中也没有找到数据文件，请确保数据文件可用。")

# 如果数据文件存在，加载它
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"成功加载数据集，共 {len(df)} 个样本")
    print("\n数据集前5行：")
    display(df.head())
else:
    print("无法加载数据集，请确保数据文件可用。")
    df = None

def create_ensemble_model(self, fold_models):
        """创建集成模型（平均所有折的预测）"""
        class EnsembleModel:
            def __init__(self, models):
                self.models = models
            
            def predict(self, x, verbose=0):
                # 确保输入是正确的形状
                if x.ndim == 1:
                    x = x.reshape(1, -1)
                elif x.ndim > 2:
                    # 如果是3D数组，reshape到2D
                    x = x.reshape(x.shape[0], -1)
                
                predictions = []
                for model in self.models:
                    pred = model.predict(x, verbose=0)
                    predictions.append(pred)
                
                # 转换为numpy数组并计算平均值
                predictions = np.array(predictions)  # shape: (num_models, batch_size, output_dim)
                mean_pred = np.mean(predictions, axis=0)  # shape: (batch_size, output_dim)
                
                return mean_pred
            
            def summary(self):
                print(f"集成模型包含 {len(self.models)} 个子模型")
                if len(self.models) > 0:
                    self.models[0].summary()
        
        return EnsembleModel(fold_models)

### 数据预处理

现在我们需要提取合金成分信息和目标属性（弹性模量和屈服强度）。

In [ ]:
import json

def extract_composition_and_phases(row):
    """
    从材料ID中提取元素成分，并从phases列中提取化合物比例
    
    示例：从 "Co8.50Mo5.15Ti2.60" 提取 [8.50, 5.15, 2.60, 83.75]
    从phases列提取化合物比例
    """
    sid = row['sid']
    phases_str = row.get('phases', '{}')  # 假设第4列名为phases
    
    # 提取元素成分（前3种元素）
    elements = ['Co', 'Mo', 'Ti']
    values = []
    
    for element in elements:
        if element in sid:
            pos = sid.find(element) + len(element)
            next_pos = len(sid)
            for next_elem in elements:
                if next_elem != element and sid.find(next_elem, pos) != -1:
                    next_pos = min(next_pos, sid.find(next_elem, pos))
            value = float(sid[pos:next_pos])
            values.append(value)
        else:
            values.append(0.0)
    
    # 计算Fe含量（余量）
    fe_content = 100.0 - sum(values)
    values.append(fe_content)
    
    # 解析化合物比例
    try:
        if isinstance(phases_str, str):
            phases_dict = json.loads(phases_str.replace("'", '"'))
        else:
            phases_dict = phases_str if isinstance(phases_str, dict) else {}
    except (json.JSONDecodeError, AttributeError):
        phases_dict = {}
    
    # 标准化化合物名称并提取比例
    phase_names = ['martensite', 'Fe2Mo', 'austenite', 'gamma_phase', 'Ni3Ti']
    phase_values = []
    
    for phase_name in phase_names:
        phase_values.append(phases_dict.get(phase_name, 0.0))
    
    return values, phase_values

def expand_composition_to_4d(composition_3d):
    """
    将3维成分扩展为4维（添加Fe含量）
    输入：[Co, Mo, Ti] (3维)
    输出：[Co, Mo, Ti, Fe] (4维)
    """
    if len(composition_3d) == 3:
        fe_content = 100.0 - sum(composition_3d)
        return np.array([composition_3d[0], composition_3d[1], composition_3d[2], fe_content])
    elif len(composition_3d) == 4:
        return np.array(composition_3d)
    else:
        raise ValueError(f"Expected 3 or 4 dimensions, got {len(composition_3d)}")

def extract_composition(sid):
    """
    从材料ID中提取元素成分
    
    示例：从 "Co8.50Mo5.15Ti2.60" 提取 [8.50, 5.15, 2.60]
    """
    elements = ['Co', 'Mo', 'Ti']
    values = []
    
    # 提取每个元素的数值
    for element in elements:
        if element in sid:
            # 找到元素在字符串中的位置
            pos = sid.find(element) + len(element)
            # 找到下一个元素的位置或字符串结尾
            next_pos = len(sid)
            for next_elem in elements:
                if next_elem != element and sid.find(next_elem, pos) != -1:
                    next_pos = min(next_pos, sid.find(next_elem, pos))
            # 提取数值
            value = float(sid[pos:next_pos])
            values.append(value)
        else:
            values.append(0.0)
            
    return values

def parse_compound_composition(compound_str):
    """
    解析化合物比例字符串
    
    输入：'{"martensite": 0.6559944215529168, "Fe2Mo": 0.08992337723785943, ...}'
    输出：[martensite_ratio, Fe2Mo_ratio, austenite_ratio, gamma_phase_ratio, Ni3Ti_ratio]
    """
    import json
    import ast
    
    try:
        # 尝试多种解析方法
        if isinstance(compound_str, str):
            # 清理字符串格式
            compound_str = compound_str.replace("'", '"')
            try:
                compound_data = json.loads(compound_str)
            except:
                # 如果JSON解析失败，尝试使用ast.literal_eval
                compound_data = ast.literal_eval(compound_str.replace('"', "'"))
        else:
            compound_data = compound_str
            
        # 定义化合物顺序
        compound_names = ['martensite', 'Fe2Mo', 'austenite', 'gamma_phase', 'Ni3Ti']
        compound_ratios = []
        
        for compound in compound_names:
            ratio = compound_data.get(compound, 0.0)  # 如果不存在该化合物，设为0
            compound_ratios.append(ratio)
            
        return compound_ratios
        
    except Exception as e:
        print(f"解析化合物数据时出错: {e}")
        print(f"原始数据: {compound_str}")
        # 返回默认值（全零）
        return [0.0, 0.0, 0.0, 0.0, 0.0]

if df is not None:
    print("开始解析元素成分和化合物比例数据...")
    
    # 检查数据集列名
    print("数据集列名：", df.columns.tolist())
    
    # 如果第4列不叫phases，需要调整列名
    if len(df.columns) > 3:
        phase_column_name = df.columns[3]  # 第4列（索引3）
        print(f"假设第4列 '{phase_column_name}' 包含化合物比例数据")
        df['phases'] = df[phase_column_name]
    
    # 提取元素成分和化合物比例
    composition_and_phases = df.apply(extract_composition_and_phases, axis=1)
    
    # 分离元素成分和化合物比例
    element_compositions = []
    phase_compositions = []
    
    for comp, phases in composition_and_phases:
        element_compositions.append(comp)
        phase_compositions.append(phases)
    
    # 转换为numpy数组
    X_elements = np.array(element_compositions)  # 4维：Co, Mo, Ti, Fe
    X_phases = np.array(phase_compositions)     # 5维：martensite, Fe2Mo, austenite, gamma_phase, Ni3Ti
    
    # 为保持与DANTE搜索空间的兼容性，同时保存3维版本
    X_elements_3d = X_elements[:, :3]  # 仅前3个元素，用于DANTE搜索
    
    print(f"元素成分数据形状：{X_elements.shape} (4维：Co, Mo, Ti, Fe)")
    print(f"元素成分3维版本：{X_elements_3d.shape} (3维：Co, Mo, Ti)")
    print(f"化合物比例数据形状：{X_phases.shape} (5维)")
    
    # 提取每个数据点的成分值（3维：Co, Mo, Ti）- 这是搜索空间
    composition_values = df['sid'].apply(extract_composition)
    X_elements = np.array(composition_values.tolist())  # 3维元素比例
    
    # 计算Fe含量（第4个元素）
    Fe_content = 100.0 - np.sum(X_elements, axis=1)
    X_elements_with_Fe = np.column_stack([X_elements, Fe_content])  # 4维元素比例（包含Fe）
    
    # 检查数据集是否包含化合物比例信息
    if len(df.columns) > 4:  # 假设第4列（索引为3）是化合物比例
        compound_column = df.iloc[:, 3]  # 第4列（索引为3）
        print("检测到化合物比例数据...")
        
        # 解析化合物比例
        compound_compositions = []
        successful_parsing = 0
        
        for i, compound_str in enumerate(compound_column):
            compound_ratios = parse_compound_composition(compound_str)
            compound_compositions.append(compound_ratios)
            
            # 检查解析是否成功（不全为0）
            if sum(compound_ratios) > 0:
                successful_parsing += 1
                
        X_compounds = np.array(compound_compositions)  # 5维化合物比例
        
        print(f"成功解析化合物数据: {successful_parsing}/{len(df)} 条记录")
        print(f"化合物比例维度: {X_compounds.shape}")
        
        # 显示化合物统计信息
        compound_names = ['martensite', 'Fe2Mo', 'austenite', 'gamma_phase', 'Ni3Ti']
        print("\n化合物比例统计:")
        for i, name in enumerate(compound_names):
            mean_ratio = np.mean(X_compounds[:, i])
            std_ratio = np.std(X_compounds[:, i])
            nonzero_count = np.count_nonzero(X_compounds[:, i])
            print(f"  {name}: 平均={mean_ratio:.4f}, 标准差={std_ratio:.4f}, 非零样本={nonzero_count}")
            
    else:
        print("未检测到化合物比例数据，将使用传统方法...")
        X_compounds = None
    
    # 分别提取弹性模量和屈服强度作为两个独立的目标
    elastic_values = df['elastic'].values
    yield_values = df['yield'].values

    # 分别计算两个目标的标准化参数
    elastic_min = np.min(elastic_values)
    elastic_max = np.max(elastic_values)
    yield_min = np.min(yield_values)
    yield_max = np.max(yield_values)

    # 分别归一化到0-1区间
    Y_elastic = (elastic_values - elastic_min) / (elastic_max - elastic_min)
    Y_yield = (yield_values - yield_min) / (yield_max - yield_min)
    
    # 计算均值用于后续摘要
    elastic_mean = np.mean(elastic_values)
    yield_mean = np.mean(yield_values)
    
    # 为向后兼容，保留组合目标Y（两个归一化值的平均）
    Y = (Y_elastic + Y_yield) / 2
    
    print("\n数据处理摘要：")
    print(f"3维元素输入维度: {X_elements.shape} (蒙特卡罗搜索空间)")
    print(f"4维元素输入维度: {X_elements_with_Fe.shape} (包含Fe，用于神经网络)")
    if X_compounds is not None:
        print(f"化合物输入维度: {X_compounds.shape}")
        print(f"总特征维度: {X_elements_with_Fe.shape[1] + X_compounds.shape[1]} (4元素+5化合物)")
    print(f"弹性模量目标维度: {Y_elastic.shape}")
    print(f"屈服强度目标维度: {Y_yield.shape}")
    print(f"组合目标维度: {Y.shape}")
    print(f"弹性模量范围: {elastic_min:.2e} - {elastic_max:.2e} (平均: {elastic_mean:.2e})")
    print(f"屈服强度范围: {yield_min:.2f} - {yield_max:.2f} (平均: {yield_mean:.2f})")
    
    # 显示成分范围
    print("\n元素成分范围：")
    print(f"Co: {X_elements[:, 0].min():.2f} to {X_elements[:, 0].max():.2f}")
    print(f"Mo: {X_elements[:, 1].min():.2f} to {X_elements[:, 1].max():.2f}")
    print(f"Ti: {X_elements[:, 2].min():.2f} to {X_elements[:, 2].max():.2f}")
    print(f"Fe (calculated): {Fe_content.min():.2f} to {Fe_content.max():.2f}")
    
    # 绘制数据分布（包括化合物数据）
    if X_compounds is not None:
        plt.figure(figsize=(20, 12))
        
        # 元素分布
        for i, element in enumerate(['Co', 'Mo', 'Ti', 'Fe']):
            plt.subplot(3, 5, i+1)
            data = X_elements_with_Fe[:, i]
            plt.hist(data, bins=20, alpha=0.7)
            plt.title(f'{element} Content Distribution')
            plt.xlabel(f'{element} Content (%)')
            plt.ylabel('Frequency')
            
        # 化合物分布
        for i, compound in enumerate(compound_names):
            plt.subplot(3, 5, i+6)
            data = X_compounds[:, i]
            # 只显示非零值
            nonzero_data = data[data > 0]
            if len(nonzero_data) > 0:
                plt.hist(nonzero_data, bins=20, alpha=0.7)
                plt.title(f'{compound} Ratio Distribution')
                plt.xlabel(f'{compound} Ratio')
                plt.ylabel('Frequency')
            else:
                plt.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=plt.gca().transAxes)
                plt.title(f'{compound} Ratio Distribution')
        
        # 性能分布
        plt.subplot(3, 5, 11)
        plt.hist(Y_elastic, bins=20, alpha=0.7, color='blue')
        plt.title('Normalized Elastic Modulus')
        plt.xlabel('Normalized Elastic Modulus')
        plt.ylabel('Frequency')
        
        plt.subplot(3, 5, 12)
        plt.hist(Y_yield, bins=20, alpha=0.7, color='red')
        plt.title('Normalized Yield Strength')
        plt.xlabel('Normalized Yield Strength')
        plt.ylabel('Frequency')
        
        plt.subplot(3, 5, 13)
        plt.hist(Y, bins=20, alpha=0.7, color='green')
        plt.title('Combined Performance')
        plt.xlabel('Combined Normalized Performance')
        plt.ylabel('Frequency')
        
        # 化合物相关性分析
        plt.subplot(3, 5, 14)
        if np.sum(X_compounds) > 0:
            compound_sums = np.sum(X_compounds, axis=1)
            plt.hist(compound_sums, bins=20, alpha=0.7, color='purple')
            plt.title('Total Compound Ratios')
            plt.xlabel('Sum of Compound Ratios')
            plt.ylabel('Frequency')
            plt.axvline(x=1.0, color='red', linestyle='--', label='Expected Sum=1')
            plt.legend()
        
        # 元素vs性能相关性
        plt.subplot(3, 5, 15)
        correlation_data = pd.DataFrame({
            'Co': X_elements_with_Fe[:, 0],
            'Mo': X_elements_with_Fe[:, 1],
            'Ti': X_elements_with_Fe[:, 2],
            'Fe': X_elements_with_Fe[:, 3],
            'Performance': Y
        })
        
        corr_matrix = correlation_data.corr()
        sns.heatmap(corr_matrix['Performance'].drop('Performance').to_frame().T, 
                   annot=True, cmap='coolwarm', center=0, cbar_kws={'label': 'Correlation'})
        plt.title('Element-Performance Correlation')
        
        plt.tight_layout()
        plt.show()
        
    else:
        # 原始绘图代码（如果没有化合物数据）
        plt.figure(figsize=(15, 10))
        
        plt.subplot(2, 3, 1)
        plt.hist(X_elements[:, 0], bins=20, alpha=0.7)
        plt.title('Co Content Distribution')
        plt.xlabel('Co Content')
        plt.ylabel('Frequency')
        
        plt.subplot(2, 3, 2)
        plt.hist(X_elements[:, 1], bins=20, alpha=0.7)
        plt.title('Mo Content Distribution')
        plt.xlabel('Mo Content')
        plt.ylabel('Frequency')
        
        plt.subplot(2, 3, 3)
        plt.hist(X_elements[:, 2], bins=20, alpha=0.7)
        plt.title('Ti Content Distribution')
        plt.xlabel('Ti Content')
        plt.ylabel('Frequency')
        
        plt.subplot(2, 3, 4)
        plt.hist(Y_elastic, bins=20, alpha=0.7, color='blue')
        plt.title('Normalized Elastic Modulus Distribution')
        plt.xlabel('Normalized Elastic Modulus')
        plt.ylabel('Frequency')
        
        plt.subplot(2, 3, 5)
        plt.hist(Y_yield, bins=20, alpha=0.7, color='red')
        plt.title('Normalized Yield Strength Distribution')
        plt.xlabel('Normalized Yield Strength')
        plt.ylabel('Frequency')
        
        plt.subplot(2, 3, 6)
        plt.hist(Y, bins=20, alpha=0.7, color='green')
        plt.title('Combined Target Value Distribution')
        plt.xlabel('Combined Normalized Performance')
        plt.ylabel('Frequency')
        
        plt.tight_layout()
        plt.show()
    
    print("\n数据预处理完成！")
    print("✓ 成功提取4维元素成分（Co, Mo, Ti, Fe）")
    print("✓ 成功提取5维化合物比例（martensite, Fe2Mo, austenite, gamma_phase, Ni3Ti）")
    print("✓ 保持3维版本以兼容DANTE搜索空间")
    print("✓ 计算了特征与性能的相关性")

### 偏度分析

计算杨氏模量和屈服强度的偏度以及对数化后的偏度，用于评估数据分布的对称性。

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# 确保数据已加载
if 'df' in locals() and df is not None:
    # 提取杨氏模量和屈服强度数据
    elastic_modulus = df['elastic'].values
    yield_strength = df['yield'].values
    
    # 计算原始数据的偏度
    elastic_skewness = stats.skew(elastic_modulus)
    yield_skewness = stats.skew(yield_strength)
    
    # 对数变换（添加小的常数以避免log(0)）
    log_elastic = np.log(elastic_modulus + 1e-10)
    log_yield = np.log(yield_strength + 1e-10)
    
    # 计算对数化后的偏度
    log_elastic_skewness = stats.skew(log_elastic)
    log_yield_skewness = stats.skew(log_yield)
    
    # 打印结果
    print("=== 偏度分析结果 ===")
    print(f"杨氏模量 (原始数据) 偏度: {elastic_skewness:.4f}")
    print(f"屈服强度 (原始数据) 偏度: {yield_skewness:.4f}")
    print(f"杨氏模量 (对数化后) 偏度: {log_elastic_skewness:.4f}")
    print(f"屈服强度 (对数化后) 偏度: {log_yield_skewness:.4f}")
    
    print("\n=== 偏度解释 ===")
    print("偏度 > 0: 右偏分布 (长尾在右侧)")
    print("偏度 < 0: 左偏分布 (长尾在左侧)")
    print("偏度 ≈ 0: 近似正态分布")
    print("|偏度| < 0.5: 近似对称")
    print("0.5 ≤ |偏度| < 1: 中等偏斜")
    print("|偏度| ≥ 1: 高度偏斜")
    
    # 可视化分布和偏度
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 杨氏模量原始分布
    axes[0, 0].hist(elastic_modulus, bins=30, alpha=0.7, color='blue', density=True)
    axes[0, 0].axvline(np.mean(elastic_modulus), color='red', linestyle='--', label=f'Mean')
    axes[0, 0].axvline(np.median(elastic_modulus), color='green', linestyle='--', label=f'Median')
    axes[0, 0].set_title(f'杨氏模量分布\n偏度: {elastic_skewness:.4f}')
    axes[0, 0].set_xlabel('杨氏模量 (Pa)')
    axes[0, 0].set_ylabel('密度')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 屈服强度原始分布
    axes[0, 1].hist(yield_strength, bins=30, alpha=0.7, color='red', density=True)
    axes[0, 1].axvline(np.mean(yield_strength), color='red', linestyle='--', label=f'Mean')
    axes[0, 1].axvline(np.median(yield_strength), color='green', linestyle='--', label=f'Median')
    axes[0, 1].set_title(f'屈服强度分布\n偏度: {yield_skewness:.4f}')
    axes[0, 1].set_xlabel('屈服强度 (Pa)')
    axes[0, 1].set_ylabel('密度')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 杨氏模量对数分布
    axes[1, 0].hist(log_elastic, bins=30, alpha=0.7, color='lightblue', density=True)
    axes[1, 0].axvline(np.mean(log_elastic), color='red', linestyle='--', label=f'Mean')
    axes[1, 0].axvline(np.median(log_elastic), color='green', linestyle='--', label=f'Median')
    axes[1, 0].set_title(f'杨氏模量对数分布\n偏度: {log_elastic_skewness:.4f}')
    axes[1, 0].set_xlabel('log(杨氏模量)')
    axes[1, 0].set_ylabel('密度')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 屈服强度对数分布
    axes[1, 1].hist(log_yield, bins=30, alpha=0.7, color='lightcoral', density=True)
    axes[1, 1].axvline(np.mean(log_yield), color='red', linestyle='--', label=f'Mean')
    axes[1, 1].axvline(np.median(log_yield), color='green', linestyle='--', label=f'Median')
    axes[1, 1].set_title(f'屈服强度对数分布\n偏度: {log_yield_skewness:.4f}')
    axes[1, 1].set_xlabel('log(屈服强度)')
    axes[1, 1].set_ylabel('密度')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 创建偏度对比表
    skewness_data = {
        '属性': ['杨氏模量', '屈服强度'],
        '原始数据偏度': [elastic_skewness, yield_skewness],
        '对数化后偏度': [log_elastic_skewness, log_yield_skewness],
        '偏度改善': [abs(elastic_skewness) - abs(log_elastic_skewness), 
                   abs(yield_skewness) - abs(log_yield_skewness)]
    }
    
    skewness_df = pd.DataFrame(skewness_data)
    print("\n=== 偏度对比表 ===")
    print(skewness_df.to_string(index=False, float_format='%.4f'))
    
    # 判断对数变换的效果
    print("\n=== 对数变换效果评估 ===")
    for i, prop in enumerate(['杨氏模量', '屈服强度']):
        improvement = skewness_data['偏度改善'][i]
        if improvement > 0:
            print(f"{prop}: 对数变换改善了分布的对称性 (偏度减少 {improvement:.4f})")
        elif improvement < 0:
            print(f"{prop}: 对数变换使分布更加偏斜 (偏度增加 {abs(improvement):.4f})")
        else:
            print(f"{prop}: 对数变换对偏度无显著影响")
            
else:
    print("错误：数据未正确加载，请先运行数据加载部分的代码。")

### 权重函数分析

计算对数变换后再归一化的杨氏模量和屈服强度在不同归一化值处的权重。
权重函数：Weight(y) = exp(beta * (y - y_mean) / y_std)，其中 beta = 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 确保数据已加载
if 'df' in locals() and df is not None:
    # 提取杨氏模量和屈服强度数据
    elastic_modulus = df['elastic'].values
    yield_strength = df['yield'].values
    
    # 对数变换（添加小的常数以避免log(0)）
    log_elastic = np.log(elastic_modulus + 1e-10)
    log_yield = np.log(yield_strength + 1e-10)
    
    # 标准化（归一化）对数变换后的数据
    scaler_elastic = StandardScaler()
    scaler_yield = StandardScaler()
    
    normalized_log_elastic = scaler_elastic.fit_transform(log_elastic.reshape(-1, 1)).flatten()
    normalized_log_yield = scaler_yield.fit_transform(log_yield.reshape(-1, 1)).flatten()
    
    # 获取归一化后的均值和标准差（理论上应该是0和1）
    elastic_mean = np.mean(normalized_log_elastic)
    elastic_std = np.std(normalized_log_elastic)
    yield_mean = np.mean(normalized_log_yield)
    yield_std = np.std(normalized_log_yield)
    
    print("=== 归一化后的统计信息 ===")
    print(f"杨氏模量 (对数+归一化) - 均值: {elastic_mean:.6f}, 标准差: {elastic_std:.6f}")
    print(f"屈服强度 (对数+归一化) - 均值: {yield_mean:.6f}, 标准差: {yield_std:.6f}")
    
    # 定义权重函数
    def weight_function(y, y_mean, y_std, beta=1):
        """计算权重函数 Weight(y) = exp(beta * (y - y_mean) / y_std)"""
        return np.exp(beta * (y - y_mean) / y_std)
    
    # 设置beta值
    beta = 1
    
    # 计算在y=0, 0.5, 1处的权重值
    y_values = [0, 0.5, 1]
    
    print(f"\n=== 权重函数分析 (beta = {beta}) ===")
    print("权重函数：Weight(y) = exp(beta * (y - y_mean) / y_std)")
    print("\n杨氏模量权重值：")
    
    elastic_weights = []
    for y in y_values:
        weight = weight_function(y, elastic_mean, elastic_std, beta)
        elastic_weights.append(weight)
        print(f"  y = {y:3.1f}: Weight = exp({beta} * ({y} - {elastic_mean:.6f}) / {elastic_std:.6f}) = {weight:.6f}")
    
    print("\n屈服强度权重值：")
    yield_weights = []
    for y in y_values:
        weight = weight_function(y, yield_mean, yield_std, beta)
        yield_weights.append(weight)
        print(f"  y = {y:3.1f}: Weight = exp({beta} * ({y} - {yield_mean:.6f}) / {yield_std:.6f}) = {weight:.6f}")
    
    # 创建权重对比表
    weight_data = {
        '归一化值 (y)': y_values,
        '杨氏模量权重': elastic_weights,
        '屈服强度权重': yield_weights
    }
    
    weight_df = pd.DataFrame(weight_data)
    print("\n=== 权重对比表 ===")
    print(weight_df.to_string(index=False, float_format='%.6f'))
    
    # 可视化权重函数
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 创建连续的y值用于绘制权重函数曲线
    y_continuous = np.linspace(-3, 3, 1000)
    
    # 杨氏模量权重函数
    elastic_weights_continuous = weight_function(y_continuous, elastic_mean, elastic_std, beta)
    ax1.plot(y_continuous, elastic_weights_continuous, 'b-', linewidth=2, label='权重函数')
    
    # 标记特定点
    for i, y in enumerate(y_values):
        ax1.plot(y, elastic_weights[i], 'ro', markersize=8, label=f'y={y}' if i < 3 else '')
        ax1.annotate(f'({y}, {elastic_weights[i]:.3f})', 
                    (y, elastic_weights[i]), 
                    xytext=(10, 10), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax1.set_xlabel('归一化值 (y)')
    ax1.set_ylabel('权重值')
    ax1.set_title(f'杨氏模量权重函数\n(beta={beta})')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 屈服强度权重函数
    yield_weights_continuous = weight_function(y_continuous, yield_mean, yield_std, beta)
    ax2.plot(y_continuous, yield_weights_continuous, 'r-', linewidth=2, label='权重函数')
    
    # 标记特定点
    for i, y in enumerate(y_values):
        ax2.plot(y, yield_weights[i], 'ro', markersize=8, label=f'y={y}' if i < 3 else '')
        ax2.annotate(f'({y}, {yield_weights[i]:.3f})', 
                    (y, yield_weights[i]), 
                    xytext=(10, 10), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax2.set_xlabel('归一化值 (y)')
    ax2.set_ylabel('权重值')
    ax2.set_title(f'屈服强度权重函数\n(beta={beta})')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    # 分析权重函数的特性
    print("\n=== 权重函数特性分析 ===")
    print(f"1. 当 y = y_mean 时，权重 = exp(0) = 1.000")
    print(f"2. 当 y > y_mean 时，权重 > 1（增强高值样本的重要性）")
    print(f"3. 当 y < y_mean 时，权重 < 1（降低低值样本的重要性）")
    print(f"4. beta = {beta} 控制权重变化的陡峭程度")
    
    # 计算权重比值
    print("\n=== 权重比值分析 ===")
    for prop, weights in [('杨氏模量', elastic_weights), ('屈服强度', yield_weights)]:
        print(f"\n{prop}:")
        print(f"  y=1.0 相对于 y=0.0 的权重比: {weights[2]/weights[0]:.3f}")
        print(f"  y=1.0 相对于 y=0.5 的权重比: {weights[2]/weights[1]:.3f}")
        print(f"  y=0.5 相对于 y=0.0 的权重比: {weights[1]/weights[0]:.3f}")
            
else:
    print("错误：数据未正确加载，请先运行数据加载部分的代码。")